<a href="https://colab.research.google.com/github/marinaalfarias/ia940/blob/main/Sintese_Aditiva.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
#!pip install librosa==0.8.0
import librosa
from IPython.display import Audio

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Checando acesso aos arquivos-fonte (se nao houver erro, os arquivos podem ser acessados)
flt, fs = librosa.load('/content/drive/Shareddrives/(IA EG)940/Sons/flute_a3_processado.wav', sr=None, mono=True)
pia, fs = librosa.load('/content/drive/Shareddrives/(IA EG)940/Sons/piano_a3_processado.wav', sr=None, mono=True)
tbn, fs = librosa.load('/content/drive/Shareddrives/(IA EG)940/Sons/trombone_a3_processado.wav', sr=None, mono=True)

# Séries Harmônicas

In [ ]:
def serie_harmonica(fs, f0, T, a, phi):
  Ts = 1/fs
  n = np.arange(fs*T)
  xn = np.zeros_like(n).astype(float)
  for m in range(len(a)):
    xn += a[m] * np.cos(2 * np.pi * (m+1) * f0 * n * Ts + phi[m])
  return xn

fs = 44100

## Transformada de Fourier

Ao aplicarmos a FFT a um sinal sem tratamento, podemos ter um vazamento espectral causado pela discontinuidade do sinal, uma vez que a FFT assume intrisicamente que o sinal sendo analisado é um ciclo de um sinal periódico infinitamente longo.

Para reduzir este efeito, janelamos o sinal antes da análise.
No entanto, ao reduzir o vazamento espectral, o janelamento talbém resulta em uma distorção na  amplitude/energia do sinal amostrado.
Para corrigir esta distorção, devemos aplicar uma correção, que é dada pela soma dos coeficientes da função de janelamento.

No fim, multiplicamos por pois uma vez que estamos analisando o espectro unilateral, desconsiderando as frequências negativas.

In [ ]:
xn = serie_harmonica(fs, 300.3, 1, [0.5, 0.3, 0.1, 0.5], [0, 0.6, 0.2, 3])
X = np.abs(np.fft.fft(xn , len(xn)*2))/len(xn)*2
X1 = np.abs(np.fft.fft(xn * np.hanning(len(xn)), len(xn)*2))

# Ponderacao
X1 /= np.sum(np.hanning(len(xn)))/2

F = np.arange(len(X)) * fs/ len(X)
plt.figure()
plt.plot(F,X1)
plt.plot(F,X)
plt.xlim(0, 2000)
plt.ylabel('O que tem neste eixo?')
plt.xlabel('O que tem neste eixo?')
plt.show()

## Figura de Lissajous

In [ ]:
f0 = 220
xn = serie_harmonica(fs, f0, 1, [1, .9, 0.8, 0.7, 0.6, 0.5],  [np.pi/2]*6)
t = np.arange(1024)

delta = 0
xbase = np.cos(2 * np.pi * (f0+delta) * t / fs)

plt.figure()
plt.plot(xbase, xn[0:len(t)])
plt.ylabel('xbase')
plt.xlabel('xn')
plt.ylim([-3, 3])
plt.show()

In [ ]:
plt.figure()
plt.plot(xbase, pia[40000:40000+len(xbase)])
plt.ylabel('sinal')
plt.xlabel('xbase')
plt.show()


In [ ]:
#flt

In [ ]:
#tbn

## Audição

In [ ]:
Audio(data=pia, rate=fs)

## Extrair série harmônica

### Sintetizar sinal de teste

In [ ]:
f0 = 440
harmonicas_gt = [1, 0.5, 0.2, 0.1]
x_alvo = serie_harmonica(fs, f0, 1, harmonicas_gt, [1, 2, 3, 4])
Audio(data=x_alvo, rate=fs)

### Extrair parâmetros

In [ ]:
# Estimar F0
freqs = librosa.yin(x_alvo, 20, 2000, sr=fs)
f0_est = np.median(freqs)
print(f0_est)

In [ ]:
# Estimar magnitudes das harmonicas
X = np.abs(np.fft.fft(x_alvo * np.hanning(len(x_alvo)), len(x_alvo)*2))
X /= np.sum(np.hanning(len(x_alvo)))/2

k0 = int(f0_est * len(X) / fs)
delta = 4
n_harmonicas = 5
harmonicas = []
for m in range(n_harmonicas):
  h_ = np.max(X[((m+1)*k0)-delta:((m+1)*k0)+delta])
  harmonicas.append(h_)

print(harmonicas)

### Avaliação

In [ ]:
Audio(data=x_alvo, rate=fs)

In [ ]:
x_novo = serie_harmonica(fs, f0, 1, harmonicas, np.random.random(len(harmonicas)))
Audio(data=x_novo, rate=fs)

In [ ]:
plt.figure()
plt.plot(x_alvo)
plt.plot(x_novo)
plt.xlim([0, 250])
plt.show()

## Extrair série harmônica de instrumento

In [ ]:
def espectro(xn, fs = 44100):
  X = np.abs(np.fft.fft(xn * np.hanning(len(xn)), len(xn)*2))

  # Ponderacao
  X /= np.sum(np.hanning(len(xn)))/2

  F = np.arange(len(X)) * fs/ len(X)
  plt.figure()
  plt.plot(F,X)
  plt.xlim(0, 2000)
  plt.ylabel('Amplitude')
  plt.xlabel('Frequência [Hz]')
  plt.show()

### Carregar nota

In [ ]:
x_alvo = pia
print(fs, x_alvo.shape)
espectro(x_alvo,fs)

### Extrair parâmetros

In [ ]:
# Estimar F0
freqs = librosa.yin(x_alvo, 20, 2000, sr=fs)
f0_est = np.median(freqs)
print(f0_est)

f0_fft = np.argmax(np.abs(np.fft.fft(x_alvo * np.hanning(len(x_alvo)), len(x_alvo)*2))) / (len(x_alvo) *2) * fs
print(f0_fft)

In [ ]:
# Estimar magnitudes das harmonicas
X = np.abs(np.fft.fft(x_alvo * np.hanning(len(x_alvo)), len(x_alvo)*2))
X /= np.sum(np.hanning(len(x_alvo)))/2
f0=220
k0 = int(f0_est * len(X) / fs)
delta = 0.2
n_harmonicas = 15
harmonicas = []
for m in range(n_harmonicas):
  #h_ = np.max(X[((m+1)*k0)-delta:((m+1)*k0)+delta])
  h_ = np.max(X[int((m+1-delta)*k0):int((m+1+delta)*k0)])
  harmonicas.append(h_)

print(harmonicas)

### Avaliação

In [ ]:
# Estimar erro (erro quadratico medio)
eqm = 0
for i in range(len(harmonicas_gt)):
  eqm += (harmonicas_gt[i]-harmonicas[i])**2

eqm /= len(harmonicas_gt)
print("EQM:", eqm)

In [ ]:
Audio(data=x_alvo, rate=fs)

In [ ]:
x_novo = serie_harmonica(fs, f0, 5, harmonicas, np.random.random(len(harmonicas)))
Audio(data=x_novo, rate=fs)

# Envoltorias no tempo

In [ ]:
x = pia

plt.figure()
plt.plot(x)
plt.plot(x_novo)
plt.show()

## Estimação

In [ ]:
# Metodo 1: retificacao + filtro passa-baixas
import scipy.signal as sig
x_ = np.abs(x)
b, a = sig.butter(4, 20, fs=fs)
x_ = sig.lfilter(b, a, x_)

t = np.arange(len(x_))/fs

plt.figure()
plt.plot(t, x, label='Som original')
plt.plot(t, x_, label='Envoltoria estimada')
plt.xlabel('Tempo (s)')
plt.ylabel('Magnitude')
#plt.semilogy()
plt.legend()
plt.show()

In [ ]:
# Metodo 2: energia RMS quadro a quadro
xrms_ = librosa.feature.rms(y=x, frame_length=2048, hop_length=512).T

T = np.arange(len(xrms_))*512/fs

plt.figure()
plt.plot(t, x, label='Som original')
plt.plot(T, xrms_, label='Valor RMS')
plt.xlabel('Tempo (s)')
plt.ylabel('Magnitude')
#plt.semilogy()
plt.legend()
plt.show()

## Parametrização

### Ataque

In [ ]:
# Ataque
pico = np.argmax(x_)
silencio = 10**-2 # Magnitude do silencio / ajuste manual?
onset = np.sum(x_[0:pico] < silencio)

# Estimacao da curva
pa = np.polyfit(t[range(onset,pico)], x_[onset:pico], deg=1)
ataque_estimado = np.polyval(pa, t[range(onset,pico)])

plt.figure()
plt.plot(t, x_, label='Envoltoria')
plt.plot(t[pico], x_[pico], 'o', label='Ponto de maximo')
plt.plot(t[onset], x_[onset], 'o', label='Onset')
plt.plot(t[range(onset,pico)], ataque_estimado, label='Curva de ataque estimada')
plt.legend()
plt.ylabel('Magnitude')
plt.xlabel('Tempo (s)')
plt.xlim(0,0.3)
plt.show()

### Decaimento

In [ ]:
# Decaimento
pico = np.argmax(x_)
decaimento = int(fs*0.5)

# Estimacao da curva
pd = np.polyfit(t[range(pico,decaimento)], x_[pico:decaimento], deg=1)
decaimento_estimado = np.polyval(pd, t[range(pico, decaimento)])

plt.figure()
plt.plot(t, x_)
plt.plot(t[pico], x_[pico], 'o')
plt.plot(t[decaimento], x_[decaimento], 'o')
plt.plot(t[range(pico,decaimento)], decaimento_estimado)
plt.xlim(0,2.3)
plt.show()

### Sustentação

In [ ]:
# Sustain
decaimento = int(fs*0.5)
sustain = int(fs*2)

# Estimacao da curva
ps = np.polyfit(t[range(decaimento,sustain)], x_[decaimento:sustain], deg=1)
sustain_estimado = np.polyval(ps, t[range(decaimento,sustain)])

plt.figure()
plt.plot(t, x_)
plt.plot(t[decaimento], x_[decaimento], 'o')
plt.plot(t[sustain], x_[sustain], 'o')
plt.plot(t[range(decaimento,sustain)], sustain_estimado)
plt.xlim(0,2.3)
#plt.semilogy()
plt.show()

### Unindo os três...

In [ ]:
print(onset, pico, decaimento, sustain)
print(pa, pd, ps)

In [ ]:
#print(onset, pico, decaimento, sustain)
def curva(fs, pontos_relevantes, derivadas):
  curva = np.zeros(pontos_relevantes[-1]) # Base
  for i in range(len(pontos_relevantes)-1):
    curva[pontos_relevantes[i]:pontos_relevantes[i+1]] = derivadas[i]
  curva = np.cumsum(curva)/fs
  return curva

In [ ]:
env = curva(fs, [onset, pico, decaimento, sustain], [pa[0], pd[0], ps[0]])

plt.figure()
plt.plot(t, x_)
plt.plot(t[onset], x_[onset], 'o')
plt.plot(t[pico], x_[pico], 'o')
plt.plot(t[decaimento], x_[decaimento], 'o')
plt.plot(t[sustain], x_[sustain], 'o')
plt.plot(t[0:len(env)], env)
plt.xlim(0,2.3)
#plt.semilogy()
plt.show()

## Aplicação e avaliação

In [ ]:
Audio(data=x_alvo, rate=fs) # Som original

In [ ]:
x_novo = serie_harmonica(fs, f0, 8, harmonicas, np.random.random(len(harmonicas)))
x_novo *= env[0:len(x_novo)]
Audio(data=x_novo, rate=fs)

# Envoltórias por harmônica

In [ ]:
import librosa.display

x_alvo = pia
D = librosa.amplitude_to_db(np.abs(librosa.stft(x_alvo)), ref=np.max)
img = librosa.display.specshow(D, y_axis='log', x_axis='time', sr=fs)


In [ ]:
Audio(data=tbn, rate=fs)

plt.plot(tbn)
plt.xlim([85000, 90000])
plt.ylim([-0.01,0.01])

In [ ]:
n_fft=4096
hop_length=512
win_length=2048
frame_rate = fs/hop_length
X_ALVO = np.abs(librosa.stft(x_alvo, n_fft=n_fft, hop_length=hop_length, win_length=win_length))
X_ALVO /= np.sum(np.hanning(win_length))/2
print(X_ALVO.shape)

In [ ]:
freqs = librosa.yin(x_alvo, 20, 2000, sr=fs, frame_length=n_fft, hop_length=hop_length, win_length=win_length)
print(np.median(freqs))


In [ ]:
n_harmonicas = 8
delta = 0.2

f0 = np.median(freqs)
matriz_harmonicas = []
for q in range(X_ALVO.shape[1]): # Para cada frame
  harmonicas = []
  for m in range(n_harmonicas):
    #f = (m+1)*f0
    #kc = int(f * n_fft / fs)

    #if kc <= delta:
    #  kc = delta+1
    #\h_ = np.max(  X_ALVO[ kc-delta:kc +delta, q]  )

    # Para cada bloco temporal, achar o máximo de vizinhança de harmônica.
    # Isso resulta na evolução temporal de cada harmônica.
    h_ = np.max(  X_ALVO[ int((m+1-delta)*f0*n_fft/fs):int((m+1+delta)*f0*n_fft/fs), q]  )
    harmonicas.append(h_)
  matriz_harmonicas.append(harmonicas)

matriz_harmonicas=np.array(matriz_harmonicas)

print(matriz_harmonicas.shape)
for i in range(n_harmonicas):
  plt.figure(figsize=(5,2))
  plt.plot(matriz_harmonicas[:,i])
  plt.title(i+1)
  plt.show()

## Síntese com envoltórias por harmônica

In [ ]:
def upsample(x, fator):
  y = np.zeros(len(x)*fator)
  for i in range(len(y)):
    y[i] = x[int(i/fator)]
  b, a = sig.butter(4, 1/fator)
  y = sig.lfilter(b, a, y)
  return y

def serie_harmonica_envoltorias(fs, f0, hop_length, envoltorias):
  n_frames, n_harmonicas = envoltorias.shape
  t = np.arange(n_frames*hop_length)/fs
  y = np.zeros_like(t)
  for m in range(n_harmonicas):
    f = f0 * (m+1)
    print("Parcial", m+1, " Freq=", f)
    env = upsample(envoltorias[:,m], hop_length)
    parcial = np.cos(2*np.pi*f*t)
    y += parcial*env
  return y


In [ ]:
f0 = np.median(freqs)
y = serie_harmonica_envoltorias(fs, f0, hop_length, matriz_harmonicas)
f0_ = librosa.yin(y, 20, 2000, sr=fs, frame_length=n_fft, hop_length=hop_length, win_length=win_length)
print(f0, np.median(f0_))

In [ ]:
Audio(data=x_alvo, rate=fs) # Som original

In [ ]:
Audio(data=y, rate=fs)

In [ ]:
plt.figure()
plt.plot(y[fs:fs+1000])
plt.plot(x_alvo[fs:fs+1000])
plt.show()

In [ ]:
Audio(data=y, rate=fs)

# Análise e re-síntese

In [ ]:
y2 = serie_harmonica_envoltorias(fs, f0*(2**(4/12)), hop_length, matriz_harmonicas)

In [ ]:
Audio(data=y2, rate=fs)

In [ ]:
Audio(data=y+y2, rate=fs)

# Alongamento de sons

In [ ]:
x_alvo = tbn
n_fft=4096
hop_length=512
win_length=2048
frame_rate = fs/hop_length
X_ALVO = np.abs(librosa.stft(x_alvo, n_fft=n_fft, hop_length=hop_length, win_length=win_length))
X_ALVO /= np.sum(np.hanning(win_length))/2
print(X_ALVO.shape)
Audio(data=x_alvo, rate=fs)

In [ ]:
n_harmonicas = 16
delta = 0.2
f0 = np.median(freqs)
matriz_harmonicas = []
for q in range(X_ALVO.shape[1]): # Para cada frame
  harmonicas = []
  for m in range(n_harmonicas):
    h_ = np.max(  X_ALVO[ int((m+1-delta)*f0*n_fft/fs):int((m+1+delta)*f0*n_fft/fs), q])
    harmonicas.append(h_)
  matriz_harmonicas.append(harmonicas)

matriz_harmonicas=np.array(matriz_harmonicas)
print(matriz_harmonicas.shape)
#for i in range(n_harmonicas):
#  plt.figure(figsize=(5,2))
#  plt.plot(matriz_harmonicas[:,i])
#  plt.title(i+1)
#  plt.show()

In [ ]:
y = serie_harmonica_envoltorias(fs, f0, int(hop_length*2), matriz_harmonicas)

In [ ]:
Audio(data=y, rate=fs)

# Morphing entre sons

In [ ]:
x_alvo_1 = pia
x_alvo_2 = flt
n_fft=4096
hop_length=512
win_length=2048
frame_rate = fs/hop_length
X_ALVO_1 = np.abs(librosa.stft(x_alvo_1, n_fft=n_fft, hop_length=hop_length, win_length=win_length))
X_ALVO_1 /= np.sum(np.hanning(win_length))/2
X_ALVO_2 = np.abs(librosa.stft(x_alvo_2, n_fft=n_fft, hop_length=hop_length, win_length=win_length))
X_ALVO_2 /= np.sum(np.hanning(win_length))/2

In [ ]:
n_harmonicas = 16
delta = 0.2
matriz_harmonicas = []
for q in range(X_ALVO_1.shape[1]): # Para cada frame
  harmonicas = []
  for m in range(n_harmonicas):
    h_ = np.max( X_ALVO_1[ int((m+1-delta)*f0*n_fft/fs):int((m+1+delta)*f0*n_fft/fs), q])
    harmonicas.append(h_)
  matriz_harmonicas.append(harmonicas)

matriz_harmonicas_1=np.array(matriz_harmonicas)

In [ ]:
n_harmonicas = 16
delta = 0.2
matriz_harmonicas = []
for q in range(X_ALVO_2.shape[1]): # Para cada frame
  harmonicas = []
  for m in range(n_harmonicas):
    h_ = np.max(  X_ALVO_2[ int((m+1-delta)*f0*n_fft/fs):int((m+1+delta)*f0*n_fft/fs), q])

    harmonicas.append(h_)
  matriz_harmonicas.append(harmonicas)

matriz_harmonicas_2=np.array(matriz_harmonicas)

In [ ]:
print(matriz_harmonicas_1.shape, matriz_harmonicas_2.shape)

In [ ]:
matriz_harmonicas_1 = matriz_harmonicas_1[0:matriz_harmonicas_2.shape[0],:]
print(matriz_harmonicas_1.shape)

In [ ]:
def serie_harmonica_morphing(fs, f0, hop_length, envoltorias_1, envoltorias_2, fator):
  envoltorias = fator * envoltorias_1 + (1-fator) * envoltorias_2
  #Combina-se o ADSR de cada harmônico para dois instrumentos
  n_frames, n_harmonicas = envoltorias.shape
  t = np.arange(n_frames*hop_length)/fs
  y = np.zeros_like(t)
  for m in range(n_harmonicas):
    f = f0 * (m+1)
    print("Parcial", m, " Freq=", f)
    env = upsample(envoltorias[:,m], hop_length)
    parcial = np.cos(2*np.pi*f*t)
    y += parcial*env
  return y

In [ ]:
y = serie_harmonica_morphing(fs, f0, hop_length*2, matriz_harmonicas_1, matriz_harmonicas_2, 0.1)

In [ ]:
Audio(data=x_alvo_1, rate=fs)


In [ ]:
Audio(data=x_alvo_2, rate=fs)

In [ ]:
Audio(data=y, rate=fs)